# Moldova IT Park - Company Shortlisting

Select active residents with any Lang in their stack and ≥10 employees.

Pipeline: load → fetch API → normalize → EDA → filter → export.

## 1. Setup

Imports and config.

In [2]:
from __future__ import annotations

import logging
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from itertools import chain
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import pydantic
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [3]:
@dataclass(frozen=True)
class Config:
    list_url: str = "https://mitp.md/p/api/web/list/getData"
    api_url: str = "https://mitp.md/p/api/web/ipCompany/read"
    page_size: int = 100
    max_pages: int = 37
    companies_csv: Path = Path("companies.csv")
    active_csv: Path = Path("active_filtered_companies.csv")
    cache_csv: Path = Path("companies_data.csv")
    output_csv: Path = Path("filtered_companies.csv")
    max_workers: int = 10
    request_timeout: float = 15.0
    retry_total: int = 3
    retry_backoff: float = 1.0
    use_cache: bool = True


CFG = Config()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    filename="app.log",
    filemode="w",
    encoding="utf-8",
)
log = logging.getLogger("mitp")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

## 2. Scrape registry

Fetch the IT Park residents registry page by page.

In [4]:
BASE_PAYLOAD = {
    "id": "ipResidentsRegistry",
    "textFilter": "",
    "advancedFilter": [],
    "pageSize": CFG.page_size,
    "sortFields": [{"field": "residentNumber", "direction": "ASC"}],
}


def fetch_page(page_number: int) -> list[dict]:
    payload = {**BASE_PAYLOAD, "pageNumber": page_number}
    r = requests.post(CFG.list_url, json=payload, timeout=10)
    r.raise_for_status()
    return r.json().get("content", [])


def parse_company(item: dict) -> dict:
    return {
        "id": item.get("_id"),
        "name": item.get("administrator"),
        "company": item.get("companyName"),
        "email": item.get("email"),
        "is_resident": item.get("isResident"),
        "phone": item.get("phone"),
    }


if CFG.use_cache and CFG.companies_csv.exists():
    df_all = pd.read_csv(CFG.companies_csv)
    print(f"loaded {len(df_all)} from cache: {CFG.companies_csv}")
else:
    pages = (fetch_page(i) for i in range(CFG.max_pages))
    df_all = pd.DataFrame(parse_company(it) for it in chain.from_iterable(pages))
    df_all.to_csv(CFG.companies_csv, index=False)
    print(f"scraped {len(df_all)} -> {CFG.companies_csv}")

df_all.head()

scraped 3700 -> companies.csv


,id,name,company,email,is_resident,phone
0,1005600040624,OXANA COROBAN,S.C. ALFASOFT S.R.L.,office@alfasoft.md,Active,+373 69164413
1,1008600028423,BAS STRIJKER,"""ADDCODE"" S.R.L.",info@addcode.md,Active,+373 22855007
2,1009600026622_1,Dorin Grițcan,ESEMPLA SYSTEMS S.R.L.,None,Inactive,None
3,1012600018982,OLEG MACARI,"Î.M. ""FBS-GROUP"" S.R.L.",oleg.macari@fbs-g.com,Active,+373 69117011
4,1010600027328,SVETLANA BRAD,"""INVENSIS SOFT"" S.R.L.",svetlana.brad@sbs.md,Active,+373 22870103


## 3. Pre-filter

Active residents with TECH/IT/DATA in the name.

In [5]:
def filter_name_companies(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["company"].str.contains("TECH|IT|DATA", case=False, na=False)]


df_input = (
    df_all    # filter_name_companies(df_all) del filter for all companies, not only with tech in name
    .loc[lambda d: d["is_resident"] == "Active"]
    .drop(columns=["Unnamed: 0"], errors="ignore")
    .drop_duplicates(subset="id")
    .reset_index(drop=True)
)
df_input.to_csv(CFG.active_csv, index=False)

print(f"rows={len(df_input)} unique_ids={df_input['id'].nunique()}")
df_input.head()

rows=2901 unique_ids=2901


,id,name,company,email,is_resident,phone
0,1005600040624,OXANA COROBAN,S.C. ALFASOFT S.R.L.,office@alfasoft.md,Active,+373 69164413
1,1008600028423,BAS STRIJKER,"""ADDCODE"" S.R.L.",info@addcode.md,Active,+373 22855007
2,1012600018982,OLEG MACARI,"Î.M. ""FBS-GROUP"" S.R.L.",oleg.macari@fbs-g.com,Active,+373 69117011
3,1010600027328,SVETLANA BRAD,"""INVENSIS SOFT"" S.R.L.",svetlana.brad@sbs.md,Active,+373 22870103
4,1008600016211,ANTON CUCER,"""META-SISTEM"" S.R.L.",reports@meta-sistem.md,Active,+373 79168383


## 4. HTTP client

Session with retry on 5xx.

In [6]:
def _build_session(cfg: Config) -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=cfg.retry_total,
        backoff_factor=cfg.retry_backoff,
        status_forcelist=(500, 502, 503, 504),
        allowed_methods=frozenset(["POST"]),
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=cfg.max_workers, pool_maxsize=cfg.max_workers)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s


def _coerce_id(value: Any) -> str:
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


session = _build_session(CFG)


def fetch_raw(company_id: Any) -> dict:
    r = session.post(CFG.api_url, json={"id": _coerce_id(company_id)}, timeout=CFG.request_timeout)
    r.raise_for_status()
    return r.json()

## 5. Schema discovery

Build a Pydantic model from a probe response.

In [7]:
probe = fetch_raw(df_input["id"].iloc[0])
print(f"fields: {len(probe)}")
probe.keys()

fields: 31


dict_keys(['companyName', 'legalHeadquarters', 'eligibleActivities', 'yearOfFoundation', 'countryOriginOfShareCapital', 'administrator', 'phone', 'email', 'programmingLanguagesUsed', 'serviceIndustry', 'salesRevenueCategory', 'numberOfEmployeesCategory', 'joinDate', 'residentNumber', 'isResident', 'requestedRegistrationPeriod', 'searchTopicOfInterest', 'logo', 'brandName', 'mainActivities', 'companyDescription', 'companyServices', 'companyProducts', 'targetMarket', 'typeOfProjects', 'awardsCertifications', 'subIndustries', 'website', 'exportSalesCategory', 'topicOfInterest', 'id'])

In [8]:
def build_model(sample: dict) -> type[pydantic.BaseModel]:
    fields = {
        key: (Optional[type(val)] if val is not None else Optional[Any], None)
        for key, val in sample.items()
    }
    return pydantic.create_model(
        "CompaniesModel",
        __config__=pydantic.ConfigDict(extra="allow", arbitrary_types_allowed=True),
        **fields,
    )


CompaniesModel = build_model(probe)

## 6. Parallel fetch

Fetch profiles with 10 threads, cache to CSV.

In [9]:
def fetch_one(company_id: Any) -> dict:
    cid = _coerce_id(company_id)
    payload: Optional[dict] = None
    try:
        payload = fetch_raw(cid)
        return CompaniesModel(**payload).model_dump()
    except Exception as exc:
        log.warning("id=%s error=%s", cid, exc)
        if isinstance(payload, dict):
            return {**payload, "error": str(exc)}
        return {"id": cid, "error": str(exc)}


def fetch_all(ids: list, max_workers: int) -> pd.DataFrame:
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        records = list(pool.map(fetch_one, ids))
    return pd.DataFrame(records)

In [10]:
if CFG.use_cache and CFG.cache_csv.exists():
    df_raw = pd.read_csv(CFG.cache_csv)
    print(f"loaded {len(df_raw)} rows from cache: {CFG.cache_csv}")
else:
    df_raw = fetch_all(df_input["id"].tolist(), max_workers=CFG.max_workers)
    df_raw.to_csv(CFG.cache_csv, index=False)
    print(f"fetched {len(df_raw)} rows -> {CFG.cache_csv}")

df_raw.head(3)

fetched 2901 rows -> companies_data.csv


,companyName,legalHeadquarters,eligibleActivities,yearOfFoundation,countryOriginOfShareCapital,administrator,phone,email,programmingLanguagesUsed,serviceIndustry,salesRevenueCategory,numberOfEmployeesCategory,joinDate,residentNumber,isResident,requestedRegistrationPeriod,searchTopicOfInterest,logo,brandName,mainActivities,companyDescription,companyServices,companyProducts,targetMarket,typeOfProjects,awardsCertifications,subIndustries,website,exportSalesCategory,topicOfInterest,id,error
0,S.C. ALFASOFT S.R.L.,"CHIŞINĂU BUIUCANI, mun. Chişinău, Alba-Iulia, ...",[62.01],2005-09-07,"{'type': 'local', 'values': [{'country': 'Repu...",OXANA COROBAN,+373 69164413,office@alfasoft.md,"[, others]","[education, eGovernment]",9-25,10-49,2018-01-17,1,Active,2027-12-31,"[62.01, softwaredev, softwaredevelopment, eGov...",{'url': '/public/images/alfasoft.png'},AlfaSoft,J6201,Dezvoltarea softului la comada,[productAndServiceDevelopment],Dezvoltarea softului la comada in baza propriu...,"{'type': 'mix', 'values': [{'country': 'USA', ...",[B2B],[-],[softwareDevelopment],-,0-25%,"[eGovernment, education]",1005600040624,NaN
1,"""ADDCODE"" S.R.L.","CHIȘINĂU BOTANICA, mun. Chișinău, Dacia bd., 4...",[62.01],2008-05-28,"{'type': 'foreign', 'values': [{'country': 'Ol...",BAS STRIJKER,+373 22855007,info@addcode.md,"[net, java, javaScript, angular, netCore, reac...","[automotive, healthMedical, constructionRealEs...",25-50,10-49,2018-01-17,2,Active,2037-12-31,"[62.01, softwaredev, softwaredevelopment, others]",{'url': '/public/images/Logotype.png'},Addcode,Software development and QA/Testing services,Addcode is originally a Dutch company from Dev...,[productAndServiceDevelopment],Software development and QA/Testing services,"{'type': 'foreign', 'values': [{'country': 'Ol...",[B2B],[ISO 27001 : 2013],"[applicationTesting, applicationDevelopment, s...",https://www.addcode.nl/,100%,[others],1008600028423,NaN
2,"Î.M. ""FBS-GROUP"" S.R.L.","CHIȘINĂU RÎȘCANI, mun. Chișinău, Moscova bd., ...","[62.01, 58.29, 62.03, 63.11, 62.02, 62.09]",2012-06-15,"{'type': 'foreign', 'values': [{'country': 'Uc...",OLEG MACARI,+373 69117011,oleg.macari@fbs-g.com,"[java, others]",[financeBanking],<9,10-49,2018-01-17,4,Active,2027-12-31,"[62.01, softwaredev, softwaredevelopment, 58.2...",{'url': '/public/images/FBS GROUP.png'},FBS-Group SRL,Activitati de realizare soft la comanda,Activitati de realizare soft la comanda,[productAndServiceDevelopment],Activitati de realizare soft la comanda,"{'type': 'mix', 'values': [{'country': 'Azerba...",[B2B],-,"[softwareDevelopment, dataAnalysisProcessing]",fbs-g.com,50-75%,[financeBanking],1012600018982,1 validation error for CompaniesModel\nawardsC...


## 7. Data quality

Compute the share of records with validation errors.

In [11]:
if "error" in df_raw.columns:
    n_err = df_raw["error"].notna().sum()
    print(f"validation/transport errors: {n_err} / {len(df_raw)} ({n_err / len(df_raw):.1%})")
    df_raw.loc[df_raw["error"].notna(), "error"].str.split("\n").str[0].value_counts().head(10)

validation/transport errors: 2063 / 2901 (71.1%)


## 8. Normalization

Parse list columns back into lists, lower-case.

In [12]:
import ast

LIST_COLS = [
    "eligibleActivities",
    "programmingLanguagesUsed",
    "serviceIndustry",
    "searchTopicOfInterest",
    "companyServices",
    "typeOfProjects",
    "subIndustries",
    "topicOfInterest",
    "awardsCertifications",
]


def _to_list(value: Any) -> list:
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str) and value.startswith("["):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []


df = df_raw.copy()
for col in LIST_COLS:
    if col in df.columns:
        df[col] = df[col].apply(_to_list).apply(lambda xs: [str(x).lower().strip() for x in xs if x])

df.shape

(2901, 32)

## 9. EDA

Distributions by size, language, industry.

In [13]:
# Company size distribution.
df["numberOfEmployeesCategory"].value_counts(dropna=False)

numberOfEmployeesCategory
<9        2163
10-49      344
NaN        319
50-249      65
>249        10
Name: count, dtype: int64

In [14]:
# Most common programming languages across the population.
df["programmingLanguagesUsed"].\
 explode().\
 dropna().\
 value_counts().\
 head(100)


programmingLanguagesUsed
javascript                    1331
others                        1146
sql                            937
python                         864
php                            864
html                           863
java                           805
css                            699
react                          546
typescript                     510
c                              430
net                            405
http                           320
angular                        280
jquery                         254
laravel                        242
golang                         218
csharp                         212
sass                           150
ruby                           100
netcore                         82
scala                           47
lua                             27
pl                              27
pascal                          22
perl                            18
react-native                    13
x                             

In [15]:
# Sub-industry frequency.
df["subIndustries"].explode().dropna().value_counts().head(20)

subIndustries
softwaredevelopment        1187
webdevelopment              911
applicationdevelopment      883
dataanalysisprocessing      766
others                      685
webdesign                   598
itauditconsulting           536
applicationtesting          462
erpcrm                      419
artificialintelligence      329
saas                        290
clouddatacenters            278
seo                         214
cybersecurity               179
marketplace                 160
gamesdevelopment            143
iteducation                 120
gpsnavigationgis            112
animationspecialeffects      95
3dtechnology                 95
Name: count, dtype: int64

In [16]:
# Languages × company size — useful when targeting hiring.
lang_size = (
    df[["numberOfEmployeesCategory", "programmingLanguagesUsed"]]
    .explode("programmingLanguagesUsed")
    .dropna(subset=["programmingLanguagesUsed"])
    .pivot_table(
        index="programmingLanguagesUsed",
        columns="numberOfEmployeesCategory",
        aggfunc="size",
        fill_value=0,
    )
    .sort_values(by=list(df["numberOfEmployeesCategory"].dropna().unique()), ascending=False)
)
lang_size.head(20)

numberOfEmployeesCategory,10-49,50-249,<9,>249
programmingLanguagesUsed,,,,
javascript,148,32,982,4
php,137,25,616,8
java,123,34,582,7
others,123,22,865,2
sql,114,25,660,6
html,96,23,632,4
net,82,16,263,7
css,77,19,511,4
python,77,17,630,5


## 10. Shortlist

Filter: Python + ≥10 employs.

In [17]:
TARGET_LANGUAGE = "python" # any lang
TARGET_SIZES = ("10-49", "50-249", ">249")

uses_python = df["programmingLanguagesUsed"].apply(lambda xs: TARGET_LANGUAGE in xs)
is_mid_or_large = df["numberOfEmployeesCategory"].isin(TARGET_SIZES)

shortlist = df.loc[uses_python & is_mid_or_large].copy()
print(f"shortlist: {len(shortlist)} companies")
shortlist["numberOfEmployeesCategory"].value_counts()

shortlist: 97 companies


numberOfEmployeesCategory
10-49     75
50-249    17
>249       5
Name: count, dtype: int64

In [18]:
SHORTLIST_COLS = [
    "id",
    "companyName",
    "numberOfEmployeesCategory",
    "yearOfFoundation",
    "website",
    "email",
    "phone",
    "programmingLanguagesUsed",
    "subIndustries",
    "serviceIndustry",
    "exportSalesCategory",
]

shortlist[SHORTLIST_COLS].sort_values("numberOfEmployeesCategory", ascending=False).head(100)

,id,companyName,numberOfEmployeesCategory,yearOfFoundation,website,email,phone,programmingLanguagesUsed,subIndustries,serviceIndustry,exportSalesCategory
25,1002600023471,Î.C.S. ENDAVA S.R.L.,>249,2000-10-12,https://www.endava.com/,ChisinauOfficeAssistants@endava.com,+373 22806700,"[sass, netcore, ruby, php, c, net, sql, python...",[softwaredevelopment],"[financebanking, telecommunications, healthmed...",100%
1953,1024600089589,AIDEVELOPMENT S.R.L.,>249,2024-12-19,n/a,office@aidevelopment.md,+373 69105740,"[php, sql, python, laravel, http]","[dataanalysisprocessing, artificialintelligenc...","[ecommerceretail, computerequipmentservices]",100%
42,1016600023872,KIVORK S.R.L.,>249,2016-08-01,-,info@kivork.md,+373 69587225,"[php, net, sql, python, java, javascript, lara...","[dataanalysisprocessing, erpcrm, itauditconsul...","[transport, researchdevelopment, tourismhospit...",100%
70,1015600037357,STEFANINI MOL S.R.L.,>249,2015-11-10,https://stefanini.com/en,Ion.Girleanu@stefanini.com,+373 22233003,"[netcore, ruby, php, c, net, sql, python, java...","[dataanalysisprocessing, clouddatacenters, cyb...","[others, computerequipmentservices, financeban...",100%
74,1009600020033,AMDARIS S.R.L.,>249,2009-06-08,https://amdaris.com/,peter.haheu@amdaris.com/ mariana.racovita@amd...,+373 6.9652355E7,"[php, net, sql, python, javascript, java, pl, ...","[softwaredevelopment, applicationtesting, appl...","[telecommunications, education, artsentertainm...",100%
...,...,...,...,...,...,...,...,...,...,...,...
334,1018600048839,ENTICON.APP S.R.L.,10-49,2018-11-29,https://enticon.pro/,admin@enticon.pro,+373 68576260,"[c, python, javascript, http]","[dataanalysisprocessing, softwaredevelopment]",[others],100%
316,1019600038503,MODSTUD-IT S.R.L.,10-49,2019-08-06,https://moldstud.com,vasile.crudu@moldstud.com,+373 68034879,"[php, c, java, javascript, golang, net, sql, p...","[applicationtesting, arvr, artificialintellige...","[healthmedical, media, bettinggambling, constr...",75-99%
315,1019600039407,URCHIN SYSTEMS S.R.L.,10-49,2019-08-14,https://urchinsys.com/,oignatiuc@urchinsys.com,+373 60506777,"[netcore, php, c, net, sql, python, javascript...","[softwaredevelopment, applicationtesting]","[healthmedical, transport]",100%
309,1018600006576,HELPTYPE S.R.L.,10-49,2018-02-15,https://helptype.md/,info@helptype.md,+373 22845584,"[php, c, sql, java, javascript, python, html, ...","[dataanalysisprocessing, erpcrm, cybersecurity...","[healthmedical, ecommerceretail, computerequip...",50-75%


In [19]:
shortlist[["id", "companyName", "numberOfEmployeesCategory","website", "email"]].head(100)

,id,companyName,numberOfEmployeesCategory,website,email
1,1008600028423,"""ADDCODE"" S.R.L.",10-49,https://www.addcode.nl/,info@addcode.md
11,1013600038187,SMART DATA S.R.L.,50-249,https://smartdata.solutions/,contact@smartdata.solutions
16,1012600038515,"Î.M. ""OCULEUS IT SERVICES"" S.R.L.",10-49,www.oculeus.com,adamov@oculeus.com
19,1013600013537,SOFTWARE MIND S.R.L.,50-249,https://www.codefactorygroup.com/,ghenadie.gorincioi@softwaremind.com
25,1002600023471,Î.C.S. ENDAVA S.R.L.,>249,https://www.endava.com/,ChisinauOfficeAssistants@endava.com
...,...,...,...,...,...
2273,1025600031105,"""CENTURIO IT SOLUTIONS"" S.R.L.",10-49,https://centour.io/,skaldone@gmail.com
2303,1025600039503,"""TECHNOSOFT SYSTEMS"" S.R.L.",10-49,-,natalia.zinchevici@ab.md
2428,1025600039189,"""KYPLO"" S.R.L.",10-49,-,alatimbalari@mail.ru
2460,1025600050959,"""MK Kredit Tech"" S.R.L.",10-49,www.mkkredit.md,petru.delinschi@mkkredit.md


## 11. Export

Save in csv file `filtered_companies.csv`.

In [20]:
shortlist[SHORTLIST_COLS].to_csv(CFG.output_csv, index=False)
print(f"wrote {len(shortlist)} rows -> {CFG.output_csv}")

wrote 97 rows -> filtered_companies.csv
